# Data Preprocessing

**Dataset:** TCGA-BRCA (Breast Invasive Carcinoma)  
**Source:** [Kaggle](https://www.kaggle.com/datasets/jmalagontorres/tcga-brca-survival-analysis)

**Pipeline:**
1. Download dataset from Kaggle
2. Resize patches to 224x224
3. Extract UNI2-h features (1536-dim)
4. Create train/val/test split (70/15/15)

In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import kagglehub
import warnings
warnings.filterwarnings("ignore")

# Configuration
SEED = 11
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Download Dataset

In [2]:
# Download dataset (~124GB) from Kaggle (requires kaggle API key)
path = kagglehub.dataset_download("jmalagontorres/tcga-brca-survival-analysis")
data_path = Path(path)
wsi_path = data_path / "WSIs"

# Load clinical data
df_raw = pd.read_csv(data_path / "clinical_data(labels).csv")

print(f"Dataset path: {path}")
print(f"Patients with images: {len(list(wsi_path.iterdir()))}")
print(f"Clinical data shape: {df_raw.shape}")

Dataset path: /home/koenen/.cache/kagglehub/datasets/jmalagontorres/tcga-brca-survival-analysis/versions/1
Patients with images: 1021
Clinical data shape: (1063, 26)


## 2. Resize Patches to 224x224

In [3]:
OUTPUT_DIR = DATA_DIR / "processed_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patient_folders = sorted([p for p in wsi_path.iterdir() if p.is_dir()])
total_new, total_skipped = 0, 0

for patient_folder in tqdm(patient_folders, desc="Resizing"):
    patient_id = patient_folder.name
    patient_output = OUTPUT_DIR / patient_id
    patient_output.mkdir(exist_ok=True)
    
    for img_path in patient_folder.glob("*.jpg"):
        output_path = patient_output / img_path.name
        if output_path.exists():
            total_skipped += 1
            continue
        try:
            img = Image.open(img_path).convert('RGB')
            img.resize((224, 224), Image.LANCZOS).save(output_path, 'JPEG', quality=95)
            total_new += 1
        except Exception as e:
            print(f"Error: {img_path}: {e}")

print(f"New: {total_new:,} | Skipped: {total_skipped:,} | Total patients: {len(patient_folders)}")


Resizing:   0%|                                                                                           | 0/1021 [00:00<?, ?it/s]


Resizing:   2%|█▎                                                                               | 17/1021 [00:00<00:06, 150.78it/s]


Resizing:   3%|██▌                                                                              | 33/1021 [00:00<00:06, 154.57it/s]


Resizing:   5%|███▉                                                                             | 49/1021 [00:00<00:06, 155.83it/s]


Resizing:   6%|█████▏                                                                           | 65/1021 [00:00<00:06, 148.83it/s]

...


Resizing:  95%|███████████████████████████████████████████████████████████████████████████▊    | 968/1021 [00:05<00:00, 163.74it/s]


Resizing:  97%|█████████████████████████████████████████████████████████████████████████████▎  | 986/1021 [00:05<00:00, 152.79it/s]


Resizing:  99%|██████████████████████████████████████████████████████████████████████████████▏| 1011/1021 [00:05<00:00, 175.85it/s]


Resizing: 100%|███████████████████████████████████████████████████████████████████████████████| 1021/1021 [00:05<00:00, 170.42it/s]

New: 0 | Skipped: 534,376 | Total patients: 1021


## 3. Extract UNI2-h Features

UNI2-h is a ViT-H/14 pretrained on 100M+ histopathology images.  
**Note:** Requires HuggingFace token with access to the UNI model.

In [ ]:
from huggingface_hub import login
from uni import get_encoder

# Login to HuggingFace (replace with your token)
login(token='YOUR_HF_TOKEN') 

# Load UNI2-h encoder
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, transform = get_encoder(enc_name='uni2-h', device=DEVICE)
model.eval()
print(f"UNI2-h loaded on {DEVICE}: {sum(p.numel() for p in model.parameters()):,} parameters")

UNI2-h loaded on cuda: 681,394,176 parameters


In [ ]:
FEATURES_DIR = DATA_DIR / "extracted_features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64

processed, skipped = 0, 0

with torch.no_grad():
    for patient_folder in tqdm(OUTPUT_DIR.iterdir(), desc="Extracting features"):
        if not patient_folder.is_dir():
            continue
        output_path = FEATURES_DIR / f"{patient_folder.name}.npy"
        if output_path.exists():
            skipped += 1
            continue
        
        image_files = sorted(patient_folder.glob("*.jpg"))
        if not image_files:
            continue
        
        features = []
        for i in range(0, len(image_files), BATCH_SIZE):
            batch = torch.stack([transform(Image.open(f).convert('RGB')) 
                                for f in image_files[i:i+BATCH_SIZE]]).to(DEVICE)
            features.append(model(batch).cpu().numpy())
        
        np.save(output_path, np.concatenate(features))
        processed += 1

print(f"Processed: {processed} | Skipped: {skipped}")

## 4. Create Train/Val/Test Split

In [6]:
# Filter patients with extracted features
valid_ids = [p.stem for p in FEATURES_DIR.glob("*.npy")]
df = df_raw[df_raw['bcr_patient_barcode'].isin(valid_ids)].copy()

# Remove uninformative columns
df = df.drop(columns=['radiation_therapy_NO', 'tissue_prospective_collection_indicator_YES'])

# Stratified split: 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['vital_status'], random_state=SEED)
test_df, val_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['vital_status'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 693 | Val: 149 | Test: 148


In [7]:
def build_final_df(splits):
    """Build final dataframe with renamed and scaled features."""
    dfs = []
    for split_name, split_df in splits:
        d = split_df.copy()
        d['split'] = split_name
        dfs.append(d)
    
    df = pd.concat(dfs).reset_index(drop=True)
    
    # Rename columns
    df = df.rename(columns={
        'bcr_patient_barcode': 'patient_id',
        'Time': 'time',
        'vital_status': 'event',
        'age_at_initial_pathologic_diagnosis': 'age',
        'lymph_node_examined_count': 'LN_examined',
        'breast_carcinoma_surgical_procedure_name_Lumpectomy': 'surgery_lumpectomy',
        'breast_carcinoma_surgical_procedure_name_Other': 'surgery_other',
        'breast_carcinoma_surgical_procedure_name_Simple Mastectomy': 'surgery_simple_mastectomy',
        'menopause_status_Indeterminate (neither Pre or Postmenopausal)': 'menopause_indeterminate',
        'menopause_status_Pre (<6 months since LMP AND no prior bilateral ovariectomy AND not on estrogen replacement)': 'menopause_pre',
        'pathologic_T_T1': 'T_T1', 'pathologic_T_T3': 'T_T3', 'pathologic_T_T4': 'T_T4', 'pathologic_T_TX': 'T_TX',
        'pathologic_N_N1': 'N_N1', 'pathologic_N_N2': 'N_N2', 'pathologic_N_N3': 'N_N3', 'pathologic_N_NX': 'N_NX',
        'pathologic_M_M1': 'M_M1', 'pathologic_M_MX': 'M_MX',
        'pathologic_stage_Stage I': 'stage_I', 'pathologic_stage_Stage III': 'stage_III',
        'pathologic_stage_Stage IV': 'stage_IV', 'pathologic_stage_Stage X': 'stage_X',
    })
    
    # Reorder columns
    meta_cols = ['patient_id', 'split', 'time', 'event']
    continuous_cols = ['age', 'LN_examined']
    onehot_cols = [c for c in df.columns if c not in meta_cols + continuous_cols]
    df = df[meta_cols + continuous_cols + sorted(onehot_cols)]
    
    # Scale features
    df['time'] = df['time'] / 365.25                # days -> years
    df['event'] = df['event'] - 1                   # 1/2 -> 0/1
    df['age'] = df['age'] / 100.0                   # scale age to [0,1]
    df['LN_examined'] = df['LN_examined'] / 44.0    # max LN_examined is 44
    
    # Convert bool to int
    for col in onehot_cols:
        if df[col].dtype == bool:
            df[col] = df[col].astype(int)
    
    return df

df_final = build_final_df([('train', train_df), ('val', val_df), ('test', test_df)])
df_final.to_csv(DATA_DIR / "clinical_data_split.csv", index=False)

# Summary
print("\nSplit Summary:")
print("=" * 50)
for split in ['train', 'val', 'test']:
    subset = df_final[df_final['split'] == split]
    events = subset['event'].sum()
    print(f"{split:5s}: {len(subset):4d} patients | {events:3.0f} events ({subset['event'].mean()*100:5.1f}%)")
print("=" * 50)
print(f"\nSaved to: {DATA_DIR / 'clinical_data_split.csv'}")


Split Summary:
train:  693 patients |  99 events ( 14.3%)
val  :  149 patients |  21 events ( 14.1%)
test :  148 patients |  21 events ( 14.2%)

Saved to: data/clinical_data_split.csv
